In [ ]:
# CELL 1 - Install dependencies
!pip install -q torch diffusers transformers accelerate
!pip install -q fastapi uvicorn pyngrok
!pip install -q opencv-python pillow numpy huggingface_hub
!pip install -q imageio imageio-ffmpeg

In [ ]:
# CELL 2 - Download Wan2.1 model
import os
from huggingface_hub import snapshot_download

# Token injected at push time by backend
hf_token = "__HF_TOKEN_PLACEHOLDER__"

model_id = "Wan-AI/Wan2.1-T2V-1.3B"
local_dir = "/kaggle/working/wan-model"

if not os.path.exists(local_dir):
    print("Downloading Wan2.1 model...")
    if hf_token:
        snapshot_download(repo_id=model_id, local_dir=local_dir, token=hf_token)
    else:
        snapshot_download(repo_id=model_id, local_dir=local_dir)
    print("Model downloaded successfully!")
else:
    print("Model already exists, skipping download.")

In [ ]:
# CELL 3 - Load Wan2.1 model into memory
import torch
from diffusers import WanPipeline
from diffusers.utils import export_to_video

print("Loading Wan2.1 model into GPU...")
pipe = WanPipeline.from_pretrained(
    "/kaggle/working/wan-model",
    torch_dtype=torch.float16
)
pipe = pipe.to("cuda")
pipe.enable_attention_slicing()
print("Model loaded successfully on GPU!")

In [ ]:
# CELL 4 - Start FastAPI server with actual Wan2.1 generation
import os
import threading
import uvicorn
import time
import base64
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from diffusers.utils import export_to_video

app = FastAPI()

last_activity = time.time()
IDLE_TIMEOUT = 5 * 60  # 5 minutes

class GenerateRequest(BaseModel):
    prompt: str
    scene_id: str
    account_id: str

@app.post("/generate")
def generate_video(request: GenerateRequest):
    global last_activity
    last_activity = time.time()
    print(f"Generating video for scene: {request.scene_id}")
    print(f"Prompt: {request.prompt[:100]}...")
    
    output_file = f"/kaggle/working/{request.scene_id}.mp4"
    
    try:
        # Wan2.1 se real video generate karo
        with torch.no_grad():
            output = pipe(
                prompt=request.prompt,
                num_frames=49,       # ~2 seconds at 24fps
                height=480,
                width=832,
                num_inference_steps=20,
                guidance_scale=5.0,
            )
        frames = output.frames[0]
        export_to_video(frames, output_file, fps=24)
        print(f"Video generated: {output_file}")
        
        # Base64 encode karke return karo
        with open(output_file, 'rb') as f:
            video_bytes = f.read()
        video_b64 = base64.b64encode(video_bytes).decode('utf-8')
        
        # Cleanup
        os.remove(output_file)
        
        last_activity = time.time()
        return {"status": "success", "video_data": video_b64}
        
    except Exception as e:
        print(f"Generation failed: {e}")
        raise HTTPException(status_code=500, detail=str(e))

@app.get("/health")
def health():
    return {"status": "alive", "idle_secs": int(time.time() - last_activity)}

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
print("FastAPI server started on port 8000 with Wan2.1 model!")

In [ ]:
# CELL 5 - Start ngrok tunnel & Register with backend
import os
import requests
from pyngrok import ngrok

# Tokens injected at push time by backend
ngrok_token = "__NGROK_TOKEN_PLACEHOLDER__"
kaggle_user = "__KAGGLE_USERNAME_PLACEHOLDER__"

if ngrok_token:
    ngrok.set_auth_token(ngrok_token)

public_url = ngrok.connect(8000).public_url
print(f"Ngrok Tunnel URL: {public_url}")

# Register to Modal Backend
backend_url = "https://irfangull2288--cartoon-backend-fastapi-modal-app.modal.run/session/register-url"
try:
    resp = requests.post(backend_url, json={
        "account_username": kaggle_user,
        "url": public_url
    })
    print("Backend registration response:", resp.json())
except Exception as e:
    print("Failed to register with backend:", e)

In [ ]:
# CELL 6 - Keep alive and idle checker
import time
print("Ready! Will auto-shutdown after 5 minutes of idle.")
while True:
    time.sleep(30)
    idle_secs = time.time() - last_activity
    print(f"Idle for {int(idle_secs)}s / {IDLE_TIMEOUT}s")
    if idle_secs > IDLE_TIMEOUT:
        print("Idle timeout reached. Cleanly exiting to stop Kaggle session.")
        break
